In [4]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
orders = pd.read_csv("../data/olist_orders_dataset.csv")

customers = pd.read_csv("../data/olist_customers_dataset.csv")

order_items = pd.read_csv("../data/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/olist_products_dataset.csv")

sellers = pd.read_csv("../data/olist_sellers_dataset.csv")

geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")

category_translation = pd.read_csv(
    "../data/product_category_name_translation.csv"
)

In [7]:
orders_raw = orders.copy()
customers_raw = customers.copy()
order_items_raw = order_items.copy()
payments_raw = payments.copy()
reviews_raw = reviews.copy()
products_raw = products.copy()
sellers_raw = sellers.copy()
geolocation_raw = geolocation.copy()
category_translation_raw = category_translation.copy()

In [8]:
datasets = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"{name:25} → {df.duplicated().sum()} duplicates")

orders                    → 0 duplicates
customers                 → 0 duplicates
order_items               → 0 duplicates
payments                  → 0 duplicates
reviews                   → 0 duplicates
products                  → 0 duplicates
sellers                   → 0 duplicates
geolocation               → 261831 duplicates
category_translation      → 0 duplicates


In [9]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [10]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [11]:
products["product_id"].duplicated().sum()

np.int64(0)

In [12]:
sellers["seller_id"].duplicated().sum()

np.int64(0)

In [15]:
orders = orders.drop_duplicates().copy()
customers = customers.drop_duplicates().copy()
products = products.drop_duplicates().copy()
sellers = sellers.drop_duplicates().copy()

In [14]:
payments = payments.drop_duplicates().copy()

In [17]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [18]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [19]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [21]:
payments["payment_value"] = pd.to_numeric(
    payments["payment_value"],
    errors="coerce"
)

payments["payment_installments"] = pd.to_numeric(
    payments["payment_installments"],
    errors="coerce"
)

In [23]:
order_items["price"] = pd.to_numeric(
    order_items["price"],
    errors="coerce"
)

order_items["freight_value"] = pd.to_numeric(
    order_items["freight_value"],
    errors="coerce"
)

In [24]:
product_numeric_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in product_numeric_columns:
    products[col] = pd.to_numeric(
        products[col],
        errors="coerce"
    )

In [25]:
missing_orders = orders.isnull().sum()

missing_orders = (
    missing_orders[missing_orders > 0]
    .sort_values(ascending=False)
)

missing_orders

order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

In [26]:
missing_products = products.isnull().sum()
missing_products[missing_products > 0].sort_values(ascending=False)

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [27]:
missing_reviews = reviews.isnull().sum()
missing_reviews[missing_reviews > 0].sort_values(ascending=False)

review_comment_title      87656
review_comment_message    58247
dtype: int64

In [28]:
orders.groupby("order_status")[
    "order_delivered_customer_date"
].apply(lambda x: x.isna().sum())

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64

In [29]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [30]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

In [31]:
delivered_orders.shape

(96478, 8)

In [32]:
delivered_orders["order_delivered_customer_date"].isna().sum()

np.int64(8)

In [33]:
delivered_orders[
    delivered_orders["order_delivered_customer_date"].isna()
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [34]:
delivered_orders = delivered_orders.dropna(
    subset=["order_delivered_customer_date"]
).copy()

In [35]:
orders = orders.dropna()

In [36]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

delivered_orders = delivered_orders.dropna(
    subset=["order_delivered_customer_date"]
)

In [37]:
(
    delivered_orders["order_delivered_customer_date"]
    <
    delivered_orders["order_purchase_timestamp"]
).sum()

np.int64(0)

In [38]:
delivered_orders[
    delivered_orders["order_delivered_customer_date"]
    <
    delivered_orders["order_purchase_timestamp"]
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [39]:
delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    -
    delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / (60 * 60 * 24)

In [40]:
delivered_orders["delivery_days"].describe()

count    96455.000000
mean        12.557693
std          9.545982
min          0.533414
25%          6.765498
50%         10.216227
75%         15.720145
max        209.628611
Name: delivery_days, dtype: float64

In [41]:
delivered_orders[
    delivered_orders["delivery_days"] < 0
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days


In [42]:
delivered_orders = delivered_orders[
    delivered_orders["delivery_days"] >= 0
].copy()

In [43]:
delivered_orders["delivery_days"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

count    96455.000000
mean        12.557693
std          9.545982
min          0.533414
1%           1.825023
5%           3.014991
25%          6.765498
50%         10.216227
75%         15.720145
95%         29.273896
99%         46.045586
max        209.628611
Name: delivery_days, dtype: float64

In [44]:
delivered_orders[
    delivered_orders["delivery_days"] > 100
][[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "delivery_days"
]].head(20)

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
1621,a4efaffc506a395c9cea7402b078c1e5,2018-01-17 10:50:09,2018-05-07 19:25:33,110.357917
3077,8b7fd198ad184563c231653673e75a7f,2017-11-14 10:04:27,2018-02-27 18:05:08,105.333808
3202,4f39a94d6e474819d898d6df7d394996,2017-04-28 16:28:03,2017-09-19 13:54:18,143.893229
4666,b31c7dea63bb08f8cdd1ec32514ccf0b,2017-09-26 18:35:35,2018-02-05 21:25:43,132.118148
10383,3602a80b09d914236f74c733631f3b8b,2018-01-31 20:38:38,2018-05-18 11:06:52,106.602940
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,191.463542
12813,c2a550cc5f966506b717532441c221e5,2018-01-12 15:38:34,2018-05-23 20:56:25,131.220729
13672,d8dbb44d7c5b1fd8e7f41b49e27053d7,2018-05-07 20:03:32,2018-08-20 17:14:55,104.882905
16945,cce224811ba8fea016c049693c3e0402,2018-02-28 12:30:39,2018-07-26 16:41:56,148.174502
18731,7e708aed151d6a8601ce8f2eaa712bf4,2018-06-02 18:37:14,2018-10-17 13:22:46,136.781620


In [45]:
products.isnull().sum().sort_values(ascending=False)

product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
product_id                      0
dtype: int64

In [46]:
reviews[
    ["review_comment_title", "review_comment_message"]
].isnull().sum()

review_comment_title      87656
review_comment_message    58247
dtype: int64

In [47]:
reviews["review_text"] = (
    reviews["review_comment_title"].fillna("")
    + " "
    + reviews["review_comment_message"].fillna("")
).str.strip()

In [48]:
reviews[
    ["review_comment_title",
     "review_comment_message",
     "review_text"]
].head()

,review_comment_title,review_comment_message,review_text
0,NaN,NaN,
1,NaN,NaN,
2,NaN,NaN,
3,NaN,Recebi bem antes do prazo estipulado.,Recebi bem antes do prazo estipulado.
4,NaN,Parabéns lojas lannister adorei comprar pela I...,Parabéns lojas lannister adorei comprar pela I...


In [53]:
(reviews["review_text"] == "").sum()

np.int64(56537)

In [54]:
reviews_nlp = reviews[
    reviews["review_text"].str.strip() != ""
].copy()

In [55]:
products["product_category_name"].value_counts().head(20)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
eletronicos                           517
construcao_ferramentas_construcao     400
eletrodomesticos                      370
Name: count, dtype: int64

In [56]:
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [57]:
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [58]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english'],
      dtype='str')

In [59]:
products[
    "product_category_name_english"
].isnull().sum()

np.int64(623)

In [61]:
products["product_category_name_english"] = (
    products["product_category_name_english"]
    .fillna("unknown")
)

In [62]:
def clean_text_column(df, column):
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )
    return df

In [63]:
products = clean_text_column(
    products,
    "product_category_name"
)

In [64]:
customers = clean_text_column(
    customers,
    "customer_city"
)

In [65]:
sellers = clean_text_column(
    sellers,
    "seller_city"
)

In [66]:
print("Negative prices:",
      (order_items["price"] < 0).sum())

print("Negative freight:",
      (order_items["freight_value"] < 0).sum())

Negative prices: 0
Negative freight: 0


In [67]:
for col in product_numeric_columns:
    print(
        col,
        "negative:",
        (products[col] < 0).sum()
    )

product_weight_g negative: 0
product_length_cm negative: 0
product_height_cm negative: 0
product_width_cm negative: 0


In [68]:
orders[
    ["order_id", "customer_id"]
].isnull().sum()

order_id       0
customer_id    0
dtype: int64

In [69]:
order_items[
    ["order_id", "product_id", "seller_id"]
].isnull().sum()

order_id      0
product_id    0
seller_id     0
dtype: int64

In [70]:
products[
    ["product_id"]
].isnull().sum()

product_id    0
dtype: int64

In [71]:
invalid_order_items = order_items[
    ~order_items["order_id"].isin(
        orders["order_id"]
    )
]

len(invalid_order_items)

2470

In [72]:
invalid_products = order_items[
    ~order_items["product_id"].isin(
        products["product_id"]
    )
]

len(invalid_products)

0

In [73]:
invalid_sellers = order_items[
    ~order_items["seller_id"].isin(
        sellers["seller_id"]
    )
]

len(invalid_sellers)

0

In [74]:
invalid_customers = orders[
    ~orders["customer_id"].isin(
        customers["customer_id"]
    )
]

len(invalid_customers)

0

In [75]:
delivered_orders.to_csv(
    "../data/processed/delivered_orders_clean.csv",
    index=False
)

customers.to_csv(
    "../data/processed/customers_clean.csv",
    index=False
)

order_items.to_csv(
    "../data/processed/order_items_clean.csv",
    index=False
)

payments.to_csv(
    "../data/processed/payments_clean.csv",
    index=False
)

reviews.to_csv(
    "../data/processed/reviews_clean.csv",
    index=False
)

products.to_csv(
    "../data/processed/products_clean.csv",
    index=False
)

sellers.to_csv(
    "../data/processed/sellers_clean.csv",
    index=False
)